# <font color="#418FDE" size="6.5" uppercase>**Stochastic Gradient**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Train SGD-based classifiers and regressors with appropriate losses and penalties. 
- Configure learning rates, early stopping, and regularization for stable training. 
- Use partial_fit and sparse inputs for incremental or large-scale workflows. 


## **1. SGD Estimators**

### **1.1. SGDClassifier Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_01_01.jpg?v=1783962633" width="250">



>* Linear classifier updated gradually from examples
>* Efficient for large, high-dimensional datasets

>* Feature weights adjust to reduce prediction errors
>* Repeated updates create class decision boundaries

>* Loss choices shape classifier behavior
>* Penalties reduce overfitting from noisy features



### **1.2. SGD Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_01_02.jpg?v=1783962629" width="250">



>* Efficient linear regression for large datasets
>* Learns through repeated small updates

>* Loss choice shapes regression error priorities
>* Robust losses reduce outlier influence

>* Regularization controls complexity and improves generalization
>* Penalties keep SGD regression scalable and robust



### **1.3. Loss Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_01_03.jpg?v=1783962631" width="250">



>* Loss measures prediction error during SGD training
>* Loss choice affects learning and generalization

>* Hinge loss favors margin-based classification
>* Logistic loss gives interpretable prediction confidence

>* Regression losses shape error sensitivity
>* Penalties reduce overfitting and complexity



In [ ]:
#@title Python Code - Loss Functions

# Compare SGD losses using simple gradients.
# This example avoids external machine learning libraries.
# Small synthetic data keeps execution fast.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(seed=7)

# Create two compact classification feature columns.
x_left = rng.normal(loc=-1.2, scale=0.55, size=30)
x_right = rng.normal(loc=1.2, scale=0.55, size=30)

# Combine features and margin labels.
x_values = np.concatenate([x_left, x_right])
y_values = np.concatenate([-np.ones(30), np.ones(30)])

# Validate matching one dimensional arrays.
assert x_values.shape == y_values.shape
assert x_values.ndim == 1

# Define one epoch of manual SGD.
def train_loss(loss_name, epochs=25, rate=0.08):
    weight, bias = 0.0, 0.0
    for epoch in range(epochs):
        order = rng.permutation(len(x_values))
        for index in order:
            margin = y_values[index] * (weight * x_values[index] + bias)
            if loss_name == "hinge" and margin < 1.0:
                weight += rate * y_values[index] * x_values[index]
                bias += rate * y_values[index]
            elif loss_name == "logistic":
                step = y_values[index] / (1.0 + np.exp(margin))
                weight += rate * step * x_values[index]
                bias += rate * step
    return weight, bias

# Train two SGD classifiers with different losses.
hinge_weight, hinge_bias = train_loss("hinge")
log_weight, log_bias = train_loss("logistic")

# Compute compact accuracy summaries.
hinge_pred = np.sign(hinge_weight * x_values + hinge_bias)
log_pred = np.sign(log_weight * x_values + log_bias)

# Print only short teaching summaries.
print(f"Manual NumPy SGD example, NumPy {np.__version__}")
print(f"Hinge loss accuracy: {(hinge_pred == y_values).mean():.2f}")
print(f"Logistic loss accuracy: {(log_pred == y_values).mean():.2f}")
print("Hinge emphasizes a margin; logistic gives smoother updates.")

# Plot data and learned decision scores.
grid = np.linspace(-3.0, 3.0, 120)
plt.figure(figsize=(7, 4))
plt.scatter(x_values, y_values, c=y_values, cmap="coolwarm", label="training points")
plt.plot(grid, hinge_weight * grid + hinge_bias, label="hinge score")
plt.plot(grid, log_weight * grid + log_bias, label="logistic score")
plt.axhline(0.0, color="black", linewidth=1)
plt.title("Different SGD loss functions change learned scores")
plt.xlabel("one feature value")
plt.ylabel("class label or score")
plt.legend()
plt.tight_layout()
plt.show()



## **2. Training Control**

### **2.1. Regularization Penalties**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_02_01.jpg?v=1783962622" width="250">



>* Regularization keeps SGD from chasing noise
>* Smaller weights help models generalize better

>* Smooth penalties shrink many feature weights
>* Sparse or combined penalties select key features

>* Tune penalties to balance fit and generalization
>* Validate settings with learning rate interactions



In [ ]:
#@title Python Code - Regularization Penalties

# Compare penalties during stochastic gradient training.
# Small synthetic data keeps execution fast.
# Regularization controls weight size and stability.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible binary classification data.
rng = np.random.default_rng(7)
samples, features = 160, 12
X = rng.normal(size=(samples, features))

# Build sparse true weights for teaching.
true_w = np.zeros(features)
true_w[:3] = [2.0, -1.5, 1.0]
logits = X @ true_w + rng.normal(scale=0.8, size=samples)

# Convert scores into zero-one labels.
y = (logits > 0).astype(float)
assert X.shape == (samples, features)

# Standardize features for stable updates.
means = X.mean(axis=0)
scales = X.std(axis=0) + 1e-8
X = (X - means) / scales

# Define sigmoid with clipping for safety.
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))

# Train logistic regression using simple SGD.
def train_sgd(penalty, alpha, epochs=25, lr=0.05):
    w = np.zeros(features)
    b = 0.0
    order = np.arange(samples)

    for epoch in range(epochs):
        rng.shuffle(order)
        for i in order:
            xi = X[i]
            pred = sigmoid(xi @ w + b)
            grad = (pred - y[i]) * xi

            if penalty == "l2":
                grad = grad + alpha * w
            elif penalty == "l1":
                grad = grad + alpha * np.sign(w)
            elif penalty == "elasticnet":
                grad = grad + alpha * (0.5 * w + 0.5 * np.sign(w))

            w = w - lr * grad
            b = b - lr * (pred - y[i])

    probs = sigmoid(X @ w + b)
    acc = np.mean((probs >= 0.5) == y)
    return w, acc

# Compare common regularization penalties.
settings = [("none", 0.0), ("l2", 0.05), ("l1", 0.05), ("elasticnet", 0.05)]
results = []

for penalty, alpha in settings:
    weights, accuracy = train_sgd(penalty, alpha)
    weight_norm = np.linalg.norm(weights, ord=2)
    near_zero = np.sum(np.abs(weights) < 0.05)
    results.append((penalty, accuracy, weight_norm, near_zero))

# Print a compact comparison table.
print("Penalty      Accuracy  L2_norm  Near_zero_weights")
for row in results:
    print(f"{row[0]:<11} {row[1]:>7.2f}   {row[2]:>6.2f}   {row[3]:>5}")

# Plot weight sizes for each penalty.
labels = [row[0] for row in results]
norms = [row[2] for row in results]
plt.figure(figsize=(6, 3))
plt.bar(labels, norms, color=["gray", "steelblue", "orange", "green"])
plt.ylabel("Weight L2 norm")
plt.title("Regularization penalties shrink SGD weights")
plt.tight_layout()
plt.show()



### **2.2. Learning Rate Schedules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_02_02.jpg?v=1783962624" width="250">



>* Schedules adjust update size for stable training
>* Start large, then refine with smaller steps

>* Schedules control exploration and later refinement
>* Choose schedules through validation, not intuition

>* Balance learning rate with regularization
>* Adapt to streams without noisy overreaction



In [ ]:
#@title Python Code - Learning Rate Schedules

# Compare learning rate schedules safely.
# Tiny data keeps training fast.
# Plots show stability differences.

import numpy as np
import matplotlib.pyplot as plt

# Create deterministic synthetic classification data.
rng = np.random.default_rng(7)
samples, features = 160, 2

# Generate two scaled feature columns.
X = rng.normal(size=(samples, features))
X[:, 1] *= 3.0

# Create labels from a noisy linear rule.
true_w = np.array([1.4, -0.8])
noise = rng.normal(scale=0.7, size=samples)

# Convert scores into binary targets.
scores = X @ true_w + noise
y = np.where(scores > 0.0, 1.0, -1.0)

# Standardize features for stable updates.
X = (X - X.mean(axis=0)) / X.std(axis=0)
assert X.shape == (samples, features)

# Define a small SGD training routine.
def train_schedule(schedule_name, epochs=18):
    weights = np.zeros(features)
    losses = []

    # Visit examples in deterministic shuffled order.
    for epoch in range(epochs):
        order = rng.permutation(samples)
        total_loss = 0.0

        # Update weights one example at a time.
        for step, index in enumerate(order, start=1):
            t = epoch * samples + step
            rate = 0.35 if schedule_name == "constant" else 0.35 / np.sqrt(t)

            # Compute hinge loss gradient with L2 penalty.
            margin = y[index] * np.dot(X[index], weights)
            gradient = 0.02 * weights

            if margin < 1.0:
                gradient -= y[index] * X[index]
                total_loss += 1.0 - margin

            weights -= rate * gradient

        # Store average hinge loss for this epoch.
        losses.append(total_loss / samples)

    return np.array(losses)

# Train two schedules on identical data.
constant_losses = train_schedule("constant")
decay_losses = train_schedule("decay")

# Print a compact comparison summary.
print("NumPy version:", np.__version__)
print("Constant final loss:", round(float(constant_losses[-1]), 3))
print("Decaying final loss:", round(float(decay_losses[-1]), 3))
print("Lower, smoother loss suggests more stable training.")

# Plot the learning curves for both schedules.
plt.figure(figsize=(7, 4))
plt.plot(constant_losses, marker="o", label="constant rate")
plt.plot(decay_losses, marker="o", label="decaying rate")
plt.xlabel("Epoch")
plt.ylabel("Average hinge loss")
plt.title("Learning Rate Schedules in Simple SGD")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Early Stopping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_02_03.jpg?v=1783962626" width="250">



>* Monitor validation performance to prevent overfitting
>* Stop at best validation score to save computation

>* Compare training loss with validation performance
>* Use patience before stopping noisy training

>* Balance stopping with learning rate and regularization
>* Save computation while preserving generalization



## **3. Incremental Sparse Learning**

### **3.1. Incremental Model Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_03_01.jpg?v=1783962637" width="250">



>* Learn continuously from small data batches
>* Update models without full retraining

>* Define all classes before incremental classification
>* Use balanced batches and validation monitoring

>* Adapt to changing data streams
>* Use safeguards to prevent model degradation



### **3.2. Sparse Input Training**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_03_02.jpg?v=1783962639" width="250">



>* Huge feature spaces, few active values
>* Store present features for efficient SGD

>* Update only nonzero features efficiently
>* Scale to streaming, massive feature spaces

>* Keep sparse feature mappings consistent
>* Regularize for scalable, adaptive learning



### **3.3. Online Anomaly Detection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_B/image_03_03.jpg?v=1783962642" width="250">



>* Detect unusual events as data arrives
>* Update models incrementally for real-time scale

>* Sparse features save memory and computation
>* Incremental updates scale anomaly detection streams

>* Normal behavior changes over time
>* Balance updates with expert review



# <font color="#418FDE" size="6.5" uppercase>**Stochastic Gradient**</font>


In this lecture, you learned to:
- Train SGD-based classifiers and regressors with appropriate losses and penalties. 
- Configure learning rates, early stopping, and regularization for stable training. 
- Use partial_fit and sparse inputs for incremental or large-scale workflows. 

In the next Module (Module 12), we will go over 'SVMs Kernel Ridge'